# S4 - ML distribuido con Spark MLlib (Regresion)

**Actividad:** integrar tres fuentes reales de sensores ambientales en un DataLake analitico particionado (Bronze -> Silver -> Gold), y sobre ese Gold entrenar y comparar modelos de regresion distribuida con Spark MLlib, reportando metricas iniciales (RMSE, R2, MAE).


## 1. Crear la `SparkSession`


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion4-ml-distribuido-regresion")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/02 20:53:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/02 20:53:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
ORIGEN_DATOS = "/opt/s04-ml-distribuido-regresion/data"
ARTIFACTS = "/opt/s04-ml-distribuido-regresion/artifacts"


## 2. Cargar las tres fuentes con esquema explicito

Tres archivos, cada uno con su propia estructura, extraidos previamente de tres sensores reales: campo electrico, campo magnetico, variables ambientales. Los tres comparten `FechaHora` como clave, medida minuto a minuto.


In [3]:
from pyspark.sql.types import StructType, StructField, TimestampType, DoubleType

schema_ce = StructType([
    StructField("FechaHora", TimestampType(), nullable=False),
    StructField("Valor_CE", DoubleType(), nullable=True),
])
df_ce = spark.read.csv(f"{ORIGEN_DATOS}/campo_electrico.csv", header=True, schema=schema_ce)
print(f"Campo electrico: {df_ce.count():,} registros")

schema_cm = StructType([
    StructField("FechaHora", TimestampType(), nullable=False),
    StructField("Valor_CM", DoubleType(), nullable=True),
])
df_cm = spark.read.csv(f"{ORIGEN_DATOS}/campo_magnetico.csv", header=True, schema=schema_cm)
print(f"Campo magnetico: {df_cm.count():,} registros")

schema_va = StructType([
    StructField("TempOut", DoubleType(), nullable=True),
    StructField("OutHum", DoubleType(), nullable=True),
    StructField("WindSpeed", DoubleType(), nullable=True),
    StructField("WindDir", DoubleType(), nullable=True),
    StructField("Bar", DoubleType(), nullable=True),
    StructField("Rain", DoubleType(), nullable=True),
    StructField("SolarRad", DoubleType(), nullable=True),
    StructField("UVIndex", DoubleType(), nullable=True),
    StructField("FechaHora", TimestampType(), nullable=False),
])
df_va = spark.read.csv(f"{ORIGEN_DATOS}/variables_ambientales.csv", header=True, schema=schema_va)
print(f"Variables ambientales: {df_va.count():,} registros")


Campo electrico: 186,664 registros
Campo magnetico: 525,600 registros
Variables ambientales: 708,958 registros


**Error frecuente**: la fuente original trae esta columna como `SolarRad.` (con un punto al final, tal como la exporta el equipo de medicion). `col("SolarRad.")` falla con `AnalysisException` -- Spark interpreta el punto como acceso a un campo anidado (`objeto.campo`), no como parte literal del nombre. La forma mas simple de evitarlo no es escapar el nombre en cada uso, sino no arrastrarlo: como `header=True` junto con un `schema` explicito hace que Spark ignore el texto del header para nombrar columnas, basta con declarar el nombre ya limpio (`SolarRad`, sin punto) en el `StructField` de arriba -- el resto del notebook nunca ve el nombre problematico.


## 3. Resolver duplicados de `FechaHora` en variables ambientales, antes de integrar

A diferencia de S3 (donde `customer_id` no tenia duplicados), aca la fuente de variables ambientales SI trae mas de una fila para el mismo minuto -- hay que resolverlo antes de unir, o el `join` multiplicaria filas sin que nadie lo note. Mismo patron de S3 (`Window`+`row_number()`): en vez de ordenar por `age`, se ordena por la fila con **menos nulos**, para conservar la version mas completa de cada minuto.


In [4]:
from pyspark.sql.functions import col, count as spark_count, when, lit

duplicadas = df_va.count() - df_va.dropDuplicates(["FechaHora"]).count()
print(f"Filas con FechaHora duplicada en variables ambientales: {duplicadas:,}")


[Stage 12:=====>                                                   (1 + 9) / 10]

Filas con FechaHora duplicada en variables ambientales: 181,918


In [5]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

columnas_conteo_nulos = [c for c in df_va.columns if c not in ("FechaHora", "WindDir")]

df_va_con_conteo = df_va.withColumn(
    "CantidadNulos",
    sum(when(col(c).isNull(), 1).otherwise(0) for c in columnas_conteo_nulos),
)

ventana_va = Window.partitionBy("FechaHora").orderBy(col("CantidadNulos").asc())

df_va_unico = (
    df_va_con_conteo
    .withColumn("row_num", row_number().over(ventana_va))
    .filter(col("row_num") == 1)
    .drop("row_num", "CantidadNulos")
)

print(f"Variables ambientales, un registro por minuto: {df_va_unico.count():,}")


26/09/02 20:54:00 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
[Stage 20:=============================>                            (1 + 1) / 2]

Variables ambientales, un registro por minuto: 527,040


## 4. Integrar las tres fuentes

`FechaHora` es la clave comun. El campo electrico queda como tabla principal (`left join`): interesa el periodo que ese sensor cubre, no el de los otros dos.


In [6]:
df_integrado = (
    df_ce
    .join(df_cm, on="FechaHora", how="left")
    .join(df_va_unico, on="FechaHora", how="left")
)

print(f"Integrado: {df_integrado.count():,} registros x {len(df_integrado.columns)} columnas")
df_integrado.printSchema()


26/09/02 20:54:03 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

Integrado: 186,664 registros x 11 columnas
root
 |-- FechaHora: timestamp (nullable = true)
 |-- Valor_CE: double (nullable = true)
 |-- Valor_CM: double (nullable = true)
 |-- TempOut: double (nullable = true)
 |-- OutHum: double (nullable = true)
 |-- WindSpeed: double (nullable = true)
 |-- WindDir: double (nullable = true)
 |-- Bar: double (nullable = true)
 |-- Rain: double (nullable = true)
 |-- SolarRad: double (nullable = true)
 |-- UVIndex: double (nullable = true)



## 5. Explorar nulos en la tabla integrada

Mismo control de calidad de S3 (2.2.3), aplicado a la tabla recien integrada -- un `left join` puede introducir nulos nuevos si algun `FechaHora` del campo electrico no tiene contraparte en las otras dos fuentes.


In [7]:
df_integrado.select([
    spark_count(when(col(c).isNull(), c)).alias(c) for c in df_integrado.columns
]).show(vertical=True, truncate=False)


26/09/02 20:54:08 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, WindDir, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, WindDir, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
[Stage 37:>                                                         (0 + 8) / 8]

-RECORD 0-----------
 FechaHora | 0      
 Valor_CE  | 0      
 Valor_CM  | 0      
 TempOut   | 0      
 OutHum    | 0      
 WindSpeed | 0      
 WindDir   | 186664 
 Bar       | 0      
 Rain      | 0      
 SolarRad  | 0      
 UVIndex   | 0      



## 6. Tratar `WindDir` (100% nula) y el codigo de error `99999`

`WindDir` no es un caso de "algunos nulos" como en H&M -- es una columna sin un solo valor util en las 708 958 filas originales. Una columna así no aporta nada al analisis; se descarta, no se rellena.

`Valor_CM` trae otro tipo de problema, distinto de un nulo: el sensor de campo magnetico usa `99999` como codigo de error del equipo, no como un valor fisico real. `isNull()` no lo detecta -- hay que conocer el dominio del dato para encontrarlo.


In [8]:
nulos_winddir = df_integrado.filter(col("WindDir").isNull()).count()
total = df_integrado.count()
print(f"WindDir nula: {nulos_winddir:,} de {total:,} ({nulos_winddir/total*100:.1f}%)")

df_sin_winddir = df_integrado.drop("WindDir")


26/09/02 20:54:12 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, WindDir, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, WindDir, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
26/09/02 20:54:16 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

WindDir nula: 186,664 de 186,664 (100.0%)


In [9]:
errores_cm = df_sin_winddir.filter(col("Valor_CM") == 99999).count()
print(f"Filas con codigo de error Valor_CM=99999: {errores_cm:,}")

df_limpio = df_sin_winddir.filter(col("Valor_CM") != 99999)
print(f"Filas despues de eliminar el codigo de error: {df_limpio.count():,}")


26/09/02 20:54:18 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

Filas con codigo de error Valor_CM=99999: 2,126


26/09/02 20:54:21 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

Filas despues de eliminar el codigo de error: 184,538


## 7. Confirmar ausencia de duplicados en la tabla final

El campo electrico (tabla principal) ya no tenia `FechaHora` duplicada, y las ambientales se resolvieron en el paso 3 -- confirma que el resultado del `join` tampoco los introdujo.


In [10]:
total_final = df_limpio.count()
sin_duplicar = df_limpio.dropDuplicates(["FechaHora"]).count()

print(f"Total: {total_final:,}, sin duplicar por FechaHora: {sin_duplicar:,}")
assert total_final == sin_duplicar, "Hay FechaHora duplicada en la tabla integrada final"


26/09/02 20:54:23 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
[Stage 82:=============================>                            (2 + 2) / 4]

Total: 184,538, sin duplicar por FechaHora: 184,538


## 8. Nulos finales sobre las 9 variables y filtrado

Las 9 variables (`Valor_CE`, `Valor_CM` y las 7 ambientales restantes, sin `WindDir`) son todas necesarias para el modelo de regresion de S4 -- una fila con cualquiera de ellas en nulo no sirve como entrada. A diferencia de H&M (donde rellenar `FN`/`Active` con 0 tenia sentido), aca ninguna de las 9 variables fisicas admite un relleno razonable: un `0` en `TempOut` no es "temperatura ausente", es una temperatura falsa. Se descartan las filas incompletas con `.na.drop(subset=[...])`, no se rellenan.


In [11]:
VARIABLES_9 = [
    "Valor_CE", "Valor_CM", "TempOut", "OutHum",
    "WindSpeed", "Bar", "Rain", "SolarRad", "UVIndex",
]

antes = df_limpio.count()
df_valido = df_limpio.na.drop(subset=VARIABLES_9)
despues = df_valido.count()

print(f"Filas antes: {antes:,}, despues de na.drop(subset=VARIABLES_9): {despues:,}")
print(f"Filas eliminadas por nulos en variables criticas: {antes - despues:,}")

df_valido = df_valido.cache()


26/09/02 20:54:26 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
26/09/02 20:54:28 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
[Stage 97:=============================>                            (2 + 2) / 4]

Filas antes: 184,538, despues de na.drop(subset=VARIABLES_9): 184,538
Filas eliminadas por nulos en variables criticas: 0


## 9. Escritura particionada en Parquet, por mes

H&M particiono por `club_member_status` (una columna categorica ya presente en los datos). Aca no existe una columna categorica natural -- pero `FechaHora` sí permite **derivar** una: el mes (`AnioMes`). Particionar series de tiempo por periodo (mes, dia) es el patron mas comun en almacenamiento analitico real, distinto del patron "particionar por categoria" de S3, pero basado en la misma idea: pocos valores distintos, usados seguido en filtros.


In [12]:
from pyspark.sql.functions import date_format

df_particionable = df_valido.withColumn("AnioMes", date_format(col("FechaHora"), "yyyy-MM"))

(
    df_particionable
    .repartition(4)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("AnioMes")
    .save(f"{ARTIFACTS}/campo_electrico_particionado")
)

import os
for carpeta in sorted(os.listdir(f"{ARTIFACTS}/campo_electrico_particionado")):
    print(carpeta)


26/09/02 20:54:32 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad., UVIndex, FechaHora
 Schema: TempOut, OutHum, WindSpeed, Bar, Rain, SolarRad, UVIndex, FechaHora
Expected: SolarRad but found: SolarRad.
CSV file: file:///opt/s04-ml-distribuido-regresion/data/variables_ambientales.csv
                                                                                

._SUCCESS.crc
AnioMes=2025-05
AnioMes=2025-06
AnioMes=2025-07
AnioMes=2025-08
AnioMes=2025-09
AnioMes=2025-10
AnioMes=2025-11
AnioMes=2025-12
_SUCCESS


## 10. Leer de vuelta y verificar el particionamiento


In [13]:
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/campo_electrico_particionado")
df_verificacion.printSchema()

assert df_verificacion.count() == df_particionable.count()
print(f"Verificado: {df_verificacion.count():,} filas, ida y vuelta sin perdida.")

df_verificacion.filter(col("AnioMes") == "2025-09").explain(True)


root
 |-- FechaHora: timestamp (nullable = true)
 |-- Valor_CE: double (nullable = true)
 |-- Valor_CM: double (nullable = true)
 |-- TempOut: double (nullable = true)
 |-- OutHum: double (nullable = true)
 |-- WindSpeed: double (nullable = true)
 |-- Bar: double (nullable = true)
 |-- Rain: double (nullable = true)
 |-- SolarRad: double (nullable = true)
 |-- UVIndex: double (nullable = true)
 |-- AnioMes: string (nullable = true)



Verificado: 184,538 filas, ida y vuelta sin perdida.
== Parsed Logical Plan ==
'Filter '`=`('AnioMes, 2025-09)
+- Relation [FechaHora#779,Valor_CE#780,Valor_CM#781,TempOut#782,OutHum#783,WindSpeed#784,Bar#785,Rain#786,SolarRad#787,UVIndex#788,AnioMes#789] parquet

== Analyzed Logical Plan ==
FechaHora: timestamp, Valor_CE: double, Valor_CM: double, TempOut: double, OutHum: double, WindSpeed: double, Bar: double, Rain: double, SolarRad: double, UVIndex: double, AnioMes: string
Filter (AnioMes#789 = 2025-09)
+- Relation [FechaHora#779,Valor_CE#780,Valor_CM#781,TempOut#782,OutHum#783,WindSpeed#784,Bar#785,Rain#786,SolarRad#787,UVIndex#788,AnioMes#789] parquet

== Optimized Logical Plan ==
Filter (isnotnull(AnioMes#789) AND (AnioMes#789 = 2025-09))
+- Relation [FechaHora#779,Valor_CE#780,Valor_CM#781,TempOut#782,OutHum#783,WindSpeed#784,Bar#785,Rain#786,SolarRad#787,UVIndex#788,AnioMes#789] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [FechaHora#779,Valor_CE#780,Valo

Para ver cuantas filas quedaron guardadas en cada particion (cada carpeta `AnioMes=...`), sin salir de Spark ni contar archivos a mano:


In [14]:
df_verificacion.groupBy("AnioMes").count().orderBy("AnioMes").show(truncate=False)


+-------+-----+
|AnioMes|count|
+-------+-----+
|2025-05|28107|
|2025-06|41502|
|2025-07|1409 |
|2025-08|20582|
|2025-09|27397|
|2025-10|32877|
|2025-11|32652|
|2025-12|12   |
+-------+-----+



In [15]:
df_valido.unpersist()


DataFrame[FechaHora: timestamp, Valor_CE: double, Valor_CM: double, TempOut: double, OutHum: double, WindSpeed: double, Bar: double, Rain: double, SolarRad: double, UVIndex: double]

## 11. Del DataLake Gold al modelado: reutilizar `df_verificacion`

`df_verificacion` (paso 10) ya es la salida Gold, leida y verificada -- no hace falta volver a leer el Parquet desde disco para empezar la parte de modelado. Se reutiliza directo como `df`:


In [16]:
df = df_verificacion


## 12. Preparar el vector de predictores (`VectorAssembler`)

Spark MLlib no acepta columnas sueltas como entrada de un modelo — necesita una sola columna vectorial que agrupe todos los predictores. `VectorAssembler` hace exactamente eso: toma N columnas numericas y las combina en una columna `features` de tipo `Vector`. `Valor_CE` queda fuera de los predictores: es la columna objetivo (`label`), no un dato de entrada.


In [17]:
from pyspark.ml.feature import VectorAssembler

PREDICTORES = [v for v in VARIABLES_9 if v != "Valor_CE"]
print(f"Predictores ({len(PREDICTORES)}): {PREDICTORES}")

ensamblador = VectorAssembler(inputCols=PREDICTORES, outputCol="features")
df_ml = ensamblador.transform(df).select("features", "Valor_CE")

df_ml.show(5, truncate=False)


Predictores (8): ['Valor_CM', 'TempOut', 'OutHum', 'WindSpeed', 'Bar', 'Rain', 'SolarRad', 'UVIndex']
+--------------------------------------------+--------+
|features                                    |Valor_CE|
+--------------------------------------------+--------+
|[24327.1,20.9,77.0,11.3,949.6,0.0,349.0,2.0]|-2.17   |
|[24274.8,16.1,87.0,4.8,949.4,0.0,184.0,1.5] |-3.02   |
|[24250.4,19.6,83.0,4.8,952.0,0.0,411.0,2.1] |-0.27   |
|[24280.4,15.2,88.0,0.0,950.8,0.0,0.0,0.0]   |0.12    |
|[24278.1,13.8,84.0,3.2,950.8,0.0,0.0,0.0]   |-0.86   |
+--------------------------------------------+--------+
only showing top 5 rows


## 13. Dividir en entrenamiento y prueba

Division aleatoria simple (80/20), no cronologica — a diferencia de una tarea de pronostico (S10), aqui cada fila es una observacion independiente, sin orden temporal que preservar.


In [18]:
df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)

print(f"Entrenamiento: {df_train.count():,} filas")
print(f"Prueba: {df_test.count():,} filas")


Entrenamiento: 147,943 filas


[Stage 139:=====>                                                 (1 + 10) / 11]

Prueba: 36,595 filas


## 14. Entrenar un modelo base: `LinearRegression`


In [19]:
from pyspark.ml.regression import LinearRegression

lr_base = LinearRegression(featuresCol="features", labelCol="Valor_CE")
modelo_base = lr_base.fit(df_train)

print("Coeficientes:", modelo_base.coefficients)
print("Intercepto:", modelo_base.intercept)


26/09/02 20:54:54 WARN Instrumentation: [ce2d53fd] regParam is zero, which might cause numerical instability and overfitting.
netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory
[Stage 145:====================>                                   (4 + 7) / 11]

Coeficientes: [-0.002788944930500952,0.145401072923687,0.006895259414812994,-0.07792759472452553,-0.04286953466350843,1.2398731396808425,-0.0007062868107990013,0.041606624500483594]
Intercepto: 104.71155302801812


## 15. Evaluar el modelo (RMSE, R2, MAE)


In [20]:
from pyspark.ml.evaluation import RegressionEvaluator

predicciones_base = modelo_base.transform(df_test)
predicciones_base.select("Valor_CE", "prediction").show(5)

def evaluar(predicciones, nombre):
    resultados = {}
    for metrica in ["rmse", "r2", "mae"]:
        evaluador = RegressionEvaluator(
            labelCol="Valor_CE", predictionCol="prediction", metricName=metrica
        )
        resultados[metrica.upper()] = evaluador.evaluate(predicciones)
    print(f"{nombre}: RMSE={resultados['RMSE']:.4f}  R2={resultados['R2']:.4f}  MAE={resultados['MAE']:.4f}")
    return resultados

resultados_base = evaluar(predicciones_base, "LinearRegression base")


+--------+--------------------+
|Valor_CE|          prediction|
+--------+--------------------+
|   -2.24|-0.19094105242808723|
|   -1.88|-0.16048159803641227|
|   -2.32| -0.2944109093496792|
|   -2.42| -0.3203480972033361|
|   -2.08|-0.01823941523674...|
+--------+--------------------+
only showing top 5 rows


[Stage 155:==============================>                         (6 + 5) / 11]

LinearRegression base: RMSE=0.7909  R2=0.2271  MAE=0.6087


## 16. Comparar configuraciones basicas (`regParam` / `elasticNetParam`)

El silabo pide comparar configuraciones basicas, no solo entrenar un unico modelo. `regParam` controla cuanto se penaliza la magnitud de los coeficientes (regularizacion); `elasticNetParam` mezcla penalizacion L1 (Lasso, `=1.0`) y L2 (Ridge, `=0.0`). Se prueban tres configuraciones simples, sin busqueda exhaustiva de hiperparametros (eso queda fuera del alcance de esta sesion):


In [21]:
configuraciones = [
    {"nombre": "Sin regularizacion", "regParam": 0.0, "elasticNetParam": 0.0},
    {"nombre": "Ridge (L2)", "regParam": 0.1, "elasticNetParam": 0.0},
    {"nombre": "Elastic Net (L1+L2)", "regParam": 0.1, "elasticNetParam": 0.5},
]

comparacion_configs = []
for config in configuraciones:
    lr = LinearRegression(
        featuresCol="features", labelCol="Valor_CE",
        regParam=config["regParam"], elasticNetParam=config["elasticNetParam"],
    )
    modelo = lr.fit(df_train)
    predicciones = modelo.transform(df_test)
    resultado = evaluar(predicciones, config["nombre"])
    resultado["Configuracion"] = config["nombre"]
    comparacion_configs.append(resultado)

import pandas as pd
pd.DataFrame(comparacion_configs)[["Configuracion", "RMSE", "R2", "MAE"]]


26/09/02 20:55:03 WARN Instrumentation: [ccae6ac4] regParam is zero, which might cause numerical instability and overfitting.
                                                                                

Sin regularizacion: RMSE=0.7909  R2=0.2271  MAE=0.6087


Ridge (L2): RMSE=0.7962  R2=0.2166  MAE=0.6174


Elastic Net (L1+L2): RMSE=0.8091  R2=0.1910  MAE=0.6372


,Configuracion,RMSE,R2,MAE
0,Sin regularizacion,0.790852,0.227109,0.608656
1,Ridge (L2),0.796224,0.216575,0.617418
2,Elastic Net (L1+L2),0.809133,0.190964,0.637151


## 17. Comparar con un segundo algoritmo: `RandomForestRegressor`

`LinearRegression` asume una relacion lineal entre predictores y objetivo. `RandomForestRegressor` no — captura relaciones no lineales e interacciones entre variables sin necesitar ese supuesto. Comparar ambas familias (lineal vs. arboles) es la forma mas basica de saber si la relacion real es, de entrada, aproximadamente lineal.


In [22]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features", labelCol="Valor_CE",
    numTrees=50, maxDepth=8, seed=42,
)
modelo_rf = rf.fit(df_train)
predicciones_rf = modelo_rf.transform(df_test)

resultados_rf = evaluar(predicciones_rf, "Random Forest")


26/09/02 20:55:32 WARN DAGScheduler: Broadcasting large task binary with size 1029.6 KiB
26/09/02 20:55:34 WARN DAGScheduler: Broadcasting large task binary with size 1963.7 KiB
                                                                                

Random Forest: RMSE=0.6732  R2=0.4399  MAE=0.5001


## 18. Importancia de variables: ¿todas aportan?

`RandomForestRegressor` calcula, sin costo adicional, cuanto reduce cada variable el error del modelo en promedio, a lo largo de todos sus arboles -- a diferencia de los coeficientes de `LinearRegression` (paso 14), que no son comparables entre si porque cada variable tiene una escala distinta (`Valor_CM` en miles, `Rain` entre 0 y 0.2). `featureImportances` si es directamente comparable: son proporciones que suman 1.0 entre todos los predictores.


In [23]:
importancias = list(zip(PREDICTORES, modelo_rf.featureImportances.toArray()))
importancias.sort(key=lambda x: x[1], reverse=True)

for variable, importancia in importancias:
    print(f"{variable:12s} {importancia:.4f}")


WindSpeed    0.3365
TempOut      0.2023
OutHum       0.1564
Valor_CM     0.1262
SolarRad     0.0800
Bar          0.0558
UVIndex      0.0428
Rain         0.0000


## 19. Comparacion final y seleccion


In [24]:
comparacion_final = pd.DataFrame(comparacion_configs + [
    {**resultados_rf, "Configuracion": "Random Forest"}
])[["Configuracion", "RMSE", "R2", "MAE"]]

comparacion_final.sort_values("RMSE")


,Configuracion,RMSE,R2,MAE
3,Random Forest,0.673222,0.439928,0.500095
0,Sin regularizacion,0.790852,0.227109,0.608656
1,Ridge (L2),0.796224,0.216575,0.617418
2,Elastic Net (L1+L2),0.809133,0.190964,0.637151


## 20. Guardar el modelo seleccionado

Elige, en base a la Tabla de la celda anterior, cual configuracion tuvo el mejor RMSE en tu propia corrida, y guardala. El nombre de variable `modelo_ganador` de abajo asume que fue el modelo de Random Forest — ajustalo segun tu resultado real.


In [25]:
modelo_ganador = modelo_rf  # ajusta esta linea segun tu propio resultado (celda anterior)

modelo_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_ce_regresion")
print(f"Modelo guardado en {ARTIFACTS}/modelo_ce_regresion")


Modelo guardado en /opt/s04-ml-distribuido-regresion/artifacts/modelo_ce_regresion


## 21. Documentar hallazgos y responder preguntas de reflexion

Agrega celdas markdown breves debajo de cada bloque de codigo (pasos 14-19) explicando que hiciste y que observaste — es la base directa de la evidencia tecnica que armaras en 4.3.1.

**Reflexion tecnica breve** (5 a 8 lineas): ¿qué diferencia de RMSE encontraste entre la configuracion sin regularizacion y la de Random Forest? ¿por qué `VectorAssembler` es un paso obligatorio en Spark MLlib y no en scikit-learn? ¿qué significaria un R2 cercano a 0 para este problema, y tu resultado se acerco a eso o se alejo?
